### Spark Session

In [ ]:
import os

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ.get("PATH", "")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.appName("crypto-stream")
    .config(
        "spark.jars.packages",
        "io.delta:delta-spark_2.13:4.4.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
    )
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.conf.set("spark.sql.caseSensitive", "true")

### Raw Layer

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, LongType

input_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "crypto.trades")
    .option("startingOffsets", "earliest")
    .load()
)

In [ ]:
input_df.printSchema()

In [ ]:
schema = StructType([StructField("e", StringType()), 
                        StructField("E", LongType()), 
                        StructField("s", StringType()), 
                        StructField("t", LongType()), 
                        StructField("p", StringType()), 
                        StructField("q", StringType()), 
                        StructField("T", LongType()), 
                        StructField("m", BooleanType()), 
                        StructField("M", BooleanType())])

input_parsed_df = (input_df.withColumn("json_str", F.from_json(F.col("value").cast("string"), schema))
                .select(F.col("json_str.*"), 
                        F.col("timestamp"), 
                        F.col("offset"), 
                        F.col("partition"), 
                        F.col("topic")))

input_clean_df = (
                input_parsed_df
                .withColumn("trade_id", F.col("t").cast("long"))
                .withColumn("price", F.col("p").cast("double"))
                .withColumn("quantity", F.col("q").cast("double"))
                .withColumn("is_buyer_maker", F.col("m").cast("boolean"))
                .withColumn("event_time", F.timestamp_millis(F.col("E").cast("long")))
                .withColumn("trade_time", F.timestamp_millis(F.col("T").cast("long")))
                .withColumn("symbol", F.col("s"))
                .withColumn("event_type", F.col("e"))
                .withColumn("is_best_price", F.col("M").cast("boolean"))
                .withColumn("date", F.to_date(F.col("event_time")))
                ).select("event_type", "event_time", "symbol", "trade_id", 
                        "price", "quantity", "trade_time", 
                        "is_buyer_maker", "is_best_price", "date")

In [ ]:
BASE_PATH = "data/lake"

BRONZE_PATH = BASE_PATH + "/bronze/trades"
CHECKPOINT_PATH = BASE_PATH + "/checkpoints/bronze/trades"

query = (input_clean_df.writeStream
    .format("delta")
    .partitionBy("date")
    .option("path", BRONZE_PATH)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime="30 seconds")
    .start())

### Silver Layer

In [ ]:
BASE_PATH = "data/lake"

BRONZE_PATH = BASE_PATH + "/bronze/trades"
CHECKPOINT_PATH = BASE_PATH + "/checkpoints/bronze/trades"

silver_df = spark.read.format("delta").load(BRONZE_PATH)

In [ ]:
silver_df.show(truncate = False)

# Testing

In [ ]:
input_df = spark.read.format("parquet") \
    .load(r"data/raw/yellow_2024-01.parquet")

In [ ]:
spark          # should print SparkSession, not error
input_df.printSchema()   # run BEFORE filter — if this fails, re-read input_df

In [ ]:
new_df = input_df.filter(input_df.VendorID == 2).withColumn("total_amount", input_df.total_amount / 100)

In [ ]:
new_df.explain(mode="formatted")

In [ ]:
new_df.show()

In [ ]:
new_df.explain(mode="cost")

In [ ]:
new_df.write.mode("overwrite").format("parquet").save(r"data/processed/yellow_2024-01_filtered.parquet")

In [ ]:
spark.read.format("parquet").load(r"data/processed/yellow_2024-01_filtered.parquet").show()

In [ ]:
new_df.write.format("delta").mode("overwrite").save(r"data/processed/yellow")

In [ ]:
spark.read.format("delta") \
    .option("versionAsOf", 2) \
    .load(r"data/processed/yellow").show()

## Streaming Data Pipeline

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, LongType

trade_schema = StructType([
    StructField("e", StringType()),      # event type
    StructField("s", StringType()),      # symbol
    StructField("t", LongType()),        # trade id
    StructField("p", StringType()),      # price
    StructField("q", StringType()),      # quantity
    StructField("T", LongType()),        # trade time
    StructField("m", BooleanType()),     # buyer is maker
])

trades = df.select(
    F.from_json(F.col("value").cast("string"), trade_schema).alias("trade"),
    F.col("timestamp"),
).select("trade.*", "timestamp").withColumnRenamed("e", "event_type") \
                        .withColumnRenamed("s", "symbol") \
                        .withColumnRenamed("t", "trade_id") \
                        .withColumnRenamed("p", "price") \
                        .withColumnRenamed("q", "quantity") \
                        .withColumnRenamed("T", "trade_time") \
                        .withColumnRenamed("m", "is_buyer_maker")

query = (
    trades.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .option("numRows", 2)
    .trigger(processingTime="5 seconds")
    .start()
)

4-5 tables in postgres, 2 trans table (payment tables) 500GB each, 3 slow moving category columns 2-3GB -- 5GB per day across 5 tables 

1. How to ingest them
2. Handling edge cases.

### Ingestion (Medillion Architecture)

1. Create spark session
2. spark read method jdbc spark.read.format("jdbc").load("source info") -- read all 5 tables from postgres, use indexed columns for faster processing
3. For 2 large tables, consider applying filters to reduce the shuffle size
4. Handle late arrived data with watermark (keep limit as 1 hour)
5. Store the raw data into s3 as delta table (As Bronze layer).

## Ingestion Code

In [ ]:
# Create Spark Session
 
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("raw_processing").getOrCreate()

# Read all 5 tables from postgres


df_tras_log_1 = spark.readStream.format("jdbc").option("url", "jdbc:postgresql://localhost:5432/postgres").option("dbtable", "tras_log_1").option("user", "postgres").option("password", "postgres").load()
df_tras_log_2 = spark.readStream.format("jdbc").option("url", "jdbc:postgresql://localhost:5432/postgres").option("dbtable", "tras_log_2").option("user", "postgres").option("password", "postgres").load()
df_category_1 = spark.readStream.format("jdbc").option("url", "jdbc:postgresql://localhost:5432/postgres").option("dbtable", "category_1").option("user", "postgres").option("password", "postgres").load()
df_category_2 = spark.readStream.format("jdbc").option("url", "jdbc:postgresql://localhost:5432/postgres").option("dbtable", "category_2").option("user", "postgres").option("password", "postgres").load()
df_category_3 = spark.readStream.format("jdbc").option("url", "jdbc:postgresql://localhost:5432/postgres").option("dbtable", "category_3").option("user", "postgres").option("password", "postgres").load()

df_tras_log_1.createOrReplaceTempView("category_3")

# Merge Upsert (ForeachBatch)
def batch_process(df, batch_id):
    spark.sql("""Merge into tras_log_1 
    using category_3 on tras_log_1.id = df_tras_log_1.id 
    when matched then update set tras_log_1.id = df_tras_log_1.id and tras_log_1.timestamp < df_tras_log_1.timestamp
    when not matched then insert (id, name) 
    values (df_tras_log_1.id, df_tras_log_1.name)""")

def batch_process(df, batch_id):
    spark.sql("""Merge into tras_log_1 
    using df_tras_log_1 on tras_log_1.id = df_tras_log_1.id 
    when matched then update set tras_log_1.id = df_tras_log_1.id and tras_log_1.timestamp < df_tras_log_1.timestamp
    when not matched then insert (id, name) 
    values (df_tras_log_1.id, df_tras_log_1.name)""")


df_tras_log_1.writeStream.option("foreachBatch", batch_process).watermark("timestamp", "1 hour").format("delta").option("path", "s3://raw_data/tras_log_1").option("checkpointLocation", "s3://raw_data/tras_log_1_checkpoint").start()
df_tras_log_2.writeStream.option("foreachBatch", batch_process).watermark("timestamp", "1 hour").format("delta").option("path", "s3://raw_data/tras_log_2").option("checkpointLocation", "s3://raw_data/tras_log_2_checkpoint").start()
df_category_1.writeStream.option("foreachBatch", batch_process).watermark("timestamp", "1 hour").format("delta").option("path", "s3://raw_data/category_1").option("checkpointLocation", "s3://raw_data/category_1_checkpoint").start()
df_category_2.writeStream.option("foreachBatch", batch_process).watermark("timestamp", "1 hour").format("delta").option("path", "s3://raw_data/category_2").option("checkpointLocation", "s3://raw_data/category_2_checkpoint").start()
df_category_3.writeStream.option("foreachBatch", batch_process).watermark("timestamp", "1 hour").format("delta").option("path", "s3://raw_data/category_3").option("checkpointLocation", "s3://raw_data/category_3_checkpoint").start()

In [ ]:
## batch Read (Reading Last 5 years) for category and 7 days for transaction

import time

transation_start_date = time.strftime("%Y-%m-%d", time.gmtime(time.time() - 7*24*60*60))
category_start_date = time.strftime("%Y-%m-%d", time.gmtime(time.time() - 5*365*24*60*60))


df_transation_1 = spark.read.format("jdbc").option("url", "jdbc:postgresql://localhost:5432/postgres") \
                    .option("dbtable", "transation_1").option("user", "postgres").option("password", "postgres").select("*").where(F.col("timestamp") > transation_start_date).load()
df_transation_2 = spark.read.format("jdbc").option("url", "jdbc:postgresql://localhost:5432/postgres") \
                    .option("dbtable", "transation_2").option("user", "postgres").option("password", "postgres").select("*").where(F.col("timestamp") > transation_start_date).load()
df_category_1 = spark.read.format("jdbc").option("url", "jdbc:postgresql://localhost:5432/postgres") \
                    .option("dbtable", "category_1").option("user", "postgres").option("password", "postgres").select("*").where(F.col("timestamp") > category_start_date).load()
df_category_2 = spark.read.format("jdbc").option("url", "jdbc:postgresql://localhost:5432/postgres") \
                    .option("dbtable", "category_2").option("user", "postgres").option("password", "postgres").select("*").where(F.col("timestamp") > category_start_date).load()
df_category_3 = spark.read.format("jdbc").option("url", "jdbc:postgresql://localhost:5432/postgres") \
                    .option("dbtable", "category_3").option("user", "postgres").option("password", "postgres").select("*").where(F.col("timestamp") > category_start_date).load()

# Merge upsert

def batch_process():
    spark.sql("""Merge into transation_1 
    using category_3 on transation_1.id = df_transation_1.id 
    when matched then update set transation_1.id = df_transation_1.id and transation_1.timestamp < df_transation_1.timestamp
    when not matched then insert (id, name) 
    values (df_transation_1.id, df_transation_1.name)""")


df_transation_1.write.format("delta").option("foreachBatch", batch_process).mode("overwrite").save("s3://raw_data/transation_1")
df_transation_2.write.format("delta").option("foreachBatch", batch_process).mode("overwrite").save("s3://raw_data/transation_2")
df_category_1.write.format("delta").option("foreachBatch", batch_process).mode("overwrite").save("s3://raw_data/category_1")
df_category_2.write.format("delta").option("foreachBatch", batch_process).mode("overwrite").save("s3://raw_data/category_2")
df_category_3.write.format("delta").option("foreachBatch", batch_process).mode("overwrite").save("s3://raw_data/category_3")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder.appName("raw_processing")
            .config(
                "spark.jars.packages",
                "io.delta:delta-spark_2.13:4.4.0,"
                "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
            )
            .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
            .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
            .getOrCreate())

In [ ]:
BASE_PATH = "data/lake"

BRONZE_PATH = BASE_PATH + "/bronze/trades"
CHECKPOINT_PATH = BASE_PATH + "/checkpoints/bronze/trades"

silver_df = spark.read.format("delta").load(BRONZE_PATH)

In [ ]:
silver_df.printSchema()

In [ ]:
silver_df.show(5, truncate = False)

In [ ]:
silver_df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in silver_df.columns]).show()

In [ ]:
silver_df.count() - silver_df.dropDuplicates(["trade_id"]).count()

In [ ]:
silver_df.agg(*[F.approx_count_distinct(c).alias(c) for c in silver_df.columns]).show()

In [ ]:
silver_df.describe().show()

In [ ]:
silver_df.columns

In [ ]:
silver_df.select([F.approx_count_distinct(c).alias(c) for c in silver_df.columns]).show(5)

In [ ]:
array = [1, 2, 3, 4, 5, 6, 7, 8]
string_array = [str(i) for i in array]

In [ ]:
print(array)
print(string_array)

In [ ]:
rows = silver_df.limit(20).collect()

In [ ]:
rows

In [ ]:
row_list = {row.trade_id: row.price for row in rows if row.price == 80380.01}

In [ ]:
row_list